<a href="https://colab.research.google.com/github/ayechan-12/Chindwin-River-Forecast/blob/main/Project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import requests
from datetime import datetime, timedelta
import io

# --- Setup ---
STATION_COORDS = {
    'Hkamti': {'lat': 25.98, 'lon': 95.70, 'danger_level': 1360, 'C': 0.6},
    'Homalin': {'lat': 24.87, 'lon': 94.90, 'danger_level': 2900, 'C': 0.5},
    'Phaungpy': {'lat': 24.28, 'lon': 94.75, 'danger_level': 1400, 'C': 0.5},
    'Mawlaik': {'lat': 23.63, 'lon': 94.42, 'danger_level': 2400, 'C': 0.4},
    'Kalewa': {'lat': 23.20, 'lon': 94.30, 'danger_level': 1550, 'C': 0.4},
    'Minkin': {'lat': 23.90, 'lon': 94.50, 'danger_level': 1350, 'C': 0.5},
    'Monywa': {'lat': 22.12, 'lon': 95.13, 'danger_level': 1000, 'C': 0.3}
}

def interpret_weather(wmo_code):
    if wmo_code <= 1: return "☀️"
    if wmo_code <= 3: return "⛅"
    return "🌧️"

def get_status(wl, danger):
    if wl >= danger: return "Danger"
    if wl >= danger * 0.9: return "Warning"
    if wl >= danger * 0.8: return "Alert"
    return "Normal"

def color_status(val):
    colors = {'Danger': '#FF4D4D', 'Warning': '#FFA64D', 'Alert': '#FFFF66', 'Normal': '#80FF80'}
    return f'background-color: {colors.get(val, "white")}; font-weight: bold; text-align: center;'

# --- UI Components ---
title = widgets.HTML("<h1 style='text-align: center;'>🌊 ချင်းတွင်းမြစ် ရေမှတ်ခန့်မှန်းချက် Dashboard</h1>")
file_upload = widgets.FileUpload(accept='.csv', description='Upload CSV File', multiple=False)
file_upload.layout.width = '300px'

station_inputs = {st: {
    'wl': widgets.Text(value='0.0', description='ရေမှတ်(cm):', layout=widgets.Layout(width='150px')),
    'or': widgets.Text(value='0.0', description='Obs.Rain(mm):', layout=widgets.Layout(width='150px'))
} for st in STATION_COORDS}

input_rows = [widgets.HBox([widgets.Label(st, layout=widgets.Layout(width='100px', font_weight='bold')),
                             station_inputs[st]['wl'], station_inputs[st]['or']]) for st in STATION_COORDS]

start_date = widgets.DatePicker(description='စတင်ရက်:', value=datetime.now().date())
end_date = widgets.DatePicker(description='ပြီးဆုံးရက်:', value=(datetime.now() + timedelta(days=2)).date())
btn_run = widgets.Button(description="📊 ခန့်မှန်းချက် ထုတ်ယူမည်", button_style='primary', layout=widgets.Layout(width='300px'))

# Output များ
content_output = widgets.Output()
path_output = widgets.Output()

def fetch_rainfall_forecast(station):
    lat, lon = STATION_COORDS[station]['lat'], STATION_COORDS[station]['lon']
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&daily=precipitation_sum,weather_code&forecast_days=7"
    try:
        resp = requests.get(url, timeout=5).json()['daily']
        return resp['time'], resp['precipitation_sum'], resp['weather_code']
    except: return [], [0.0]*7, [0]*7

def on_run(b):
    content_output.clear_output()
    path_output.clear_output()

    with path_output:
        if file_upload.value:
            # Handle newer ipywidgets file upload format
            fname = file_upload.value[0]['name'] if isinstance(file_upload.value, list) else list(file_upload.value.values())[0]['metadata']['name']
            print(f"📁 လက်ရှိအသုံးပြုထားသောဖိုင်: {fname}")
        else:
            print("📁 ဖိုင်တင်သွင်းထားခြင်းမရှိပါ။")

    with content_output:
        df_hist = None
        if file_upload.value:
            try:
                uploaded_file = file_upload.value[0] if isinstance(file_upload.value, list) else list(file_upload.value.values())[0]
                content = uploaded_file['content']
                df_hist = pd.read_csv(io.BytesIO(content))
                df_hist.columns = [c.strip().capitalize() for c in df_hist.columns]
                df_hist['Date'] = pd.to_datetime(df_hist['Date'])
            except: pass

        start, end = start_date.value, end_date.value
        if not start or not end or start > end: return

        all_data = []
        for st in STATION_COORDS:
            try:
                wl_initial = float(station_inputs[st]['wl'].value)
                obs_rain = float(station_inputs[st]['or'].value)
            except: wl_initial, obs_rain = 0.0, 0.0

            times, rf_fcst, weather_codes = fetch_rainfall_forecast(st)

            current_day = start
            while current_day <= end:
                try:
                    idx = (current_day - datetime.now().date()).days
                    fcst_rain = int(round(rf_fcst[idx])) if 0 <= idx < len(rf_fcst) else 0
                    w_code = weather_codes[idx] if 0 <= idx < len(weather_codes) else 0
                except: fcst_rain, w_code = 0, 0

                forecast_wl = int(round(wl_initial + ((obs_rain + fcst_rain) * STATION_COORDS[st]['C'] * 2)))
                change = forecast_wl - wl_initial
                to_danger = forecast_wl - STATION_COORDS[st]['danger_level']

                all_data.append({
                    'Day': current_day.strftime('%Y-%m-%d'), 'Date_Obj': current_day,
                    'Station': st, 'Forecast WL': forecast_wl,
                    'Change': f"{'+' if change > 0 else ''}{change}",
                    'Rain(mm)': fcst_rain,
                    'To Danger': f"{'+' if to_danger > 0 else ''}{to_danger}",
                    'Weather': interpret_weather(w_code),
                    'Status': get_status(forecast_wl, STATION_COORDS[st]['danger_level'])
                })
                current_day += timedelta(days=1)

        df = pd.DataFrame(all_data)

        # --- Graph ---
        fig, ax = plt.subplots(figsize=(10, 5))
        for st in STATION_COORDS:
            if df_hist is not None and st in df_hist.columns:
                hist_data = df_hist.tail(90)
                ax.plot(hist_data['Date'], hist_data[st], linestyle='-', linewidth=2.5, alpha=0.6, label=f"{st} (Hist)")

            subset = df[df['Station'] == st]
            ax.plot(subset['Date_Obj'], subset['Forecast WL'], marker='o', linestyle='None', markersize=5, label=f"{st} (Fcst)")

        ax.set_title('Chindwin River Water Level Forecast')
        ax.set_ylabel('Water Level (cm)')
        ax.grid(True)
        plt.xticks(rotation=45)
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()

        # Graph ကို Center ထားရန် ပြင်ဆင်ချက်
        graph_out = widgets.Output(layout=widgets.Layout(display='flex', justify_content='center'))
        with graph_out:
            plt.show()

        # --- Tables ---
        tables = []
        for day in df['Day'].unique():
            table_day = widgets.Output()
            with table_day:
                display(widgets.HTML(f"<h3>📅 {day}</h3>"))
                styled = df[df['Day'] == day].drop(columns=['Day', 'Date_Obj']).style.map(color_status, subset=['Status'])
                display(styled)
            tables.append(table_day)

        # Graph ကို Center ကျစေရန် VBox ဖြင့် စီစဉ်ခြင်း
        display(widgets.VBox([widgets.HBox(tables), graph_out]))

btn_run.on_click(on_run)

# Dashboard စီစဉ်ခြင်း
dashboard = widgets.VBox([title, file_upload, widgets.HBox([start_date, end_date]), btn_run, content_output, path_output, widgets.VBox(input_rows)])
display(dashboard)